**Minimal-Kopie mit QUANT_MODES-Filter.** Füge bei Bedarf deine Analysezellen unten an.


# QUANT\_MODES Filter hinzugefügt

Diese Notebook-Version fügt einen Filter für `quant_mode` hinzu (`no-quant`, `quant-16`, `quant-8`, `quant-8-full`).  
Der Filter wird automatisch auf **Experiment_Summary.csv** angewendet, indem `pandas.read_csv` gepatcht wird.


In [1]:

# %%
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
    _HAS_WIDGETS = True
except Exception:
    _HAS_WIDGETS = False

def _find_summary_csv():
    candidates = [
        Path("Output") / "Error_Metrics" / "Experiment_Summary.csv",
        Path(".") / "Output" / "Error_Metrics" / "Experiment_Summary.csv",
        Path("..") / "Output" / "Error_Metrics" / "Experiment_Summary.csv",
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    for root, _, files in os.walk(".", topdown=True):
        if "Experiment_Summary.csv" in files:
            return Path(root) / "Experiment_Summary.csv"
    return None

SUMMARY_CSV_PATH = _find_summary_csv()
if SUMMARY_CSV_PATH is None:
    display(Markdown("**⚠️** *Experiment_Summary.csv* nicht automatisch gefunden. "
                     "Passe bei Bedarf `SUMMARY_CSV_PATH` manuell an."))
else:
    display(Markdown(f"**Gefunden:** `{SUMMARY_CSV_PATH}`"))

_available_modes = []
try:
    if SUMMARY_CSV_PATH and Path(SUMMARY_CSV_PATH).exists():
        _df_probe = pd.read_csv(SUMMARY_CSV_PATH)
        if "quant_mode" in _df_probe.columns:
            _available_modes = sorted(list(pd.Series(_df_probe["quant_mode"].dropna().astype(str)).unique()))
except Exception as e:
    display(Markdown(f"**⚠️ Hinweis:** quant_mode-Werte nicht erkannt: `{e}`"))

_selected_modes = _available_modes.copy() if _available_modes else []

if _HAS_WIDGETS and _available_modes:
    select_all_chk = widgets.Checkbox(value=True, description="Alle Modi wählen")
    modes_select = widgets.SelectMultiple(
        options=_available_modes,
        value=tuple(_available_modes),
        description="quant_mode",
        disabled=False,
        layout=widgets.Layout(width="50%")
    )
    apply_btn = widgets.Button(description="Filter anwenden", button_style="")
    out = widgets.Output()

    def _on_select_all_change(change):
        if change["name"] == "value":
            with out:
                out.clear_output(wait=True)
                if change["new"]:
                    modes_select.value = tuple(_available_modes)
                else:
                    modes_select.value = tuple(m for m in modes_select.value)

    def _on_apply_clicked(btn):
        global _selected_modes
        with out:
            out.clear_output(wait=True)
            _selected_modes = list(modes_select.value)
            if select_all_chk.value:
                _selected_modes = _available_modes.copy()
            if not _selected_modes:
                print("Kein Modus ausgewählt – leerer Filter.")
            else:
                print("Aktive QUANT_MODES:", _selected_modes)

    select_all_chk.observe(_on_select_all_change)
    apply_btn.on_click(_on_apply_clicked)

    display(widgets.VBox([select_all_chk, modes_select, apply_btn, out]))
else:
    display(Markdown("**Widget-Fallback aktiv.** Setze `\_selected_modes` manuell, z. B.: "
                     "`_selected_modes = ['quant-16', 'quant-8']`"))
    _selected_modes = _available_modes.copy()

_pd_read_csv_orig = pd.read_csv

def _is_summary_csv_arg(arg):
    try:
        p = Path(arg)
        return p.name.lower() == "experiment_summary.csv"
    except Exception:
        return False

def _apply_quant_filter(df: pd.DataFrame) -> pd.DataFrame:
    if "quant_mode" in df.columns:
        try:
            if _selected_modes:
                return df[df["quant_mode"].astype(str).isin(_selected_modes)].copy()
        except Exception:
            return df
    return df

def _patched_read_csv(*args, **kwargs):
    df = _pd_read_csv_orig(*args, **kwargs)
    try:
        is_target = False
        if len(args) >= 1 and _is_summary_csv_arg(args[0]):
            is_target = True
        elif "filepath_or_buffer" in kwargs and _is_summary_csv_arg(kwargs["filepath_or_buffer"]):
            is_target = True
        if is_target or ("quant_mode" in getattr(df, "columns", [])):
            return _apply_quant_filter(df)
        return df
    except Exception:
        return df

pd.read_csv = _patched_read_csv

display(Markdown("✅ **QUANT\_MODES-Filter aktiv.** Ladevorgänge für *Experiment_Summary.csv* (und DataFrames mit `quant_mode`) werden gefiltert."))

def load_summary_filtered():
    if SUMMARY_CSV_PATH is None:
        raise FileNotFoundError("Pfad zu Experiment_Summary.csv unbekannt.")
    return pd.read_csv(SUMMARY_CSV_PATH)

try:
    _df_preview = load_summary_filtered().head(10)
    display(Markdown("**Vorschau (gefiltert):**"))
    display(_df_preview)
except Exception as e:
    display(Markdown(f"**Hinweis:** Vorschau nicht verfügbar: `{e}`"))


**Gefunden:** `C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Output\Error_Metrics\Experiment_Summary.csv`

**Widget-Fallback aktiv.** Setze `\_selected_modes` manuell, z. B.: `_selected_modes = ['quant-16', 'quant-8']`

✅ **QUANT\_MODES-Filter aktiv.** Ladevorgänge für *Experiment_Summary.csv* (und DataFrames mit `quant_mode`) werden gefiltert.

**Vorschau (gefiltert):**

,algorithm,profile,lags,horizon,model_variant,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,run_id
0,light_xgboost,edge,1,1,model.joblib,3.868195,40.779835,24.656584,71.931034,0.1724,2025-08-27_220542_3110_train
1,light_xgboost,edge,1,4,model.joblib,11.482434,47.815058,27.843396,53.317241,0.6854,2025-08-27_220631_8665_train
2,light_xgboost,edge,1,7,model.joblib,20.970045,57.558049,30.941960,49.500000,1.2024,2025-08-27_220726_9653_train
3,light_xgboost,edge,1,10,model.joblib,31.156471,68.052612,31.594643,49.600000,1.7079,2025-08-27_220815_4267_train
4,light_xgboost,edge,1,13,model.joblib,36.020204,73.917982,32.181781,48.441379,2.2245,2025-08-27_220903_3999_train
5,light_xgboost,edge,1,16,model.joblib,42.779825,79.387491,33.677106,49.100000,2.7333,2025-08-27_220952_4675_train
6,light_xgboost,edge,4,1,model.joblib,3.013478,48.582342,24.603468,47.300000,0.1765,2025-08-27_221043_1735_train
7,light_xgboost,edge,4,4,model.joblib,13.213362,59.459166,27.014802,48.000000,0.6994,2025-08-27_221129_4495_train
8,light_xgboost,edge,4,7,model.joblib,20.370912,65.313337,29.155195,48.300000,1.2078,2025-08-27_221215_1460_train
9,light_xgboost,edge,4,10,model.joblib,29.361122,74.413263,30.693209,48.700000,1.7296,2025-08-27_221303_5772_train
